# A2d paired Pedestrian localization-loss gate

This is a short controlled gate, not four full runs. It starts four branches from frozen A2 epoch 130 and changes only the matched Pedestrian 2D box L1 and GIoU weight: control 1.0, then 1.5, 2.0, and 2.5. Each branch fine-tunes for five epochs at 1e-5. The gate evaluates complete KITTI AP, nearby recall, and corrected M44 localization failures before authorizing at most one full run. Use a Colab GPU and run top-to-bottom.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, os, shlex, shutil, subprocess, sys
MOBILE_REPO=Path('/content/mobile_adas3d'); MONODETR_REPO=Path('/content/MonoDETR')
MONODETR_COMMIT='6994b9f512400b258c6edb75f77423beb9c126f2'
DRIVE_DATASET_ROOT=Path('/content/drive/MyDrive/datasets/kitti'); LOCAL_DATASET_ROOT=Path('/content/kitti')
SPLIT_DIR=Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen')
DATASET_ROOT=Path('/content/monodetr_kitti_a2d')
R0_SELECTION=Path('/content/drive/MyDrive/mobile_adas3d_outputs/references/monodetr_r0/product_checkpoint_sweep/r0_product_selection.json')
A2_SELECTION=Path('/content/drive/MyDrive/mobile_adas3d_outputs/students/monodetr_a2_gt/product_checkpoint_sweep/a2_product_selection.json')
OUTPUT_ROOT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/students/monodetr_a2d_pedestrian_box_gate')
def run(command,cwd=None,env=None):
    command=[str(x) for x in command]; print('+',shlex.join(command),flush=True)
    merged=os.environ.copy(); merged.update(env or {})
    result=subprocess.run(command,cwd=cwd,env=merged)
    if result.returncode: raise RuntimeError(f'Exit {result.returncode}: {shlex.join(command)}')
run(['nvidia-smi'])


In [ ]:
# Fetch pinned sources and install the exact compatibility/model/loss patches.
if not MOBILE_REPO.exists(): run(['git','clone','https://github.com/ali-rt/mobile_adas3d.git',MOBILE_REPO])
else: run(['git','pull','--ff-only'],cwd=MOBILE_REPO)
if not MONODETR_REPO.exists(): run(['git','clone','https://github.com/ZrrSkywalker/MonoDETR.git',MONODETR_REPO])
run(['git','fetch','--all'],cwd=MONODETR_REPO); run(['git','checkout',MONODETR_COMMIT],cwd=MONODETR_REPO)
run([sys.executable,'-m','pip','install','-q','gdown','pyyaml','scipy','opencv-python-headless','numba','scikit-image','tqdm','ninja','timm==1.0.20','pandas'])
for patch in ('patch_monodetr_colab_compat.py','patch_monodetr_product_taxonomy.py','patch_monodetr_mobilenetv4.py','patch_monodetr_verbose_resume.py','patch_monodetr_checkpoint_metadata.py','patch_monodetr_pedestrian_box_weight.py'):
    run([sys.executable,f'scripts/{patch}','--monodetr-repo',MONODETR_REPO],cwd=MOBILE_REPO)
ops=MONODETR_REPO/'lib/models/monodetr/ops'; shutil.rmtree(ops/'build',ignore_errors=True)
run([sys.executable,'setup.py','build','install'],cwd=ops,env={'MAX_JOBS':'2'})
run([sys.executable,'-c','import torch, timm, MultiScaleDeformableAttention; from lib.models.monodetr import build_monodetr; print(torch.__version__,timm.__version__,torch.cuda.get_device_name(0))'],cwd=MONODETR_REPO)


In [ ]:
# Recreate an isolated canonical Chen-split KITTI view.
def resolve(root,names):
    for name in names:
        path=root/name
        if path.is_dir(): return path
sources={}
for key,names in {'image_2':['training/image_2','training/image_02'],'label_2':['training/label_2','training/label_02'],'calib':['training/calib']}.items():
    sources[key]=resolve(LOCAL_DATASET_ROOT,names) or resolve(DRIVE_DATASET_ROOT,names)
if any(path is None for path in sources.values()): raise FileNotFoundError(sources)
(DATASET_ROOT/'training').mkdir(parents=True,exist_ok=True); (DATASET_ROOT/'ImageSets').mkdir(parents=True,exist_ok=True)
for name,target in sources.items():
    link=DATASET_ROOT/'training'/name
    if link.is_symlink() and link.resolve()==target.resolve(): continue
    if link.exists() or link.is_symlink(): raise RuntimeError(f'Refusing to replace {link}')
    link.symlink_to(target,target_is_directory=True)
for split in ('train','val'): shutil.copy2(SPLIT_DIR/f'{split}.txt',DATASET_ROOT/'ImageSets'/f'{split}.txt')
assert len((DATASET_ROOT/'ImageSets/train.txt').read_text().splitlines())==3712
assert len((DATASET_ROOT/'ImageSets/val.txt').read_text().splitlines())==3769


In [ ]:
# Build the frozen A2 base config, verify epoch 130, and emit all four paired gate configs.
for required in (R0_SELECTION,A2_SELECTION):
    if not required.is_file(): raise FileNotFoundError(required)
BASE_CONFIG=MONODETR_REPO/'configs/monodetr_a2d_base.yaml'
run([sys.executable,'scripts/prepare_monodetr_a2_student.py','--monodetr-repo',MONODETR_REPO,'--dataset-root',DATASET_ROOT,'--r0-selection',R0_SELECTION,'--output-root','/content/a2d_base','--config-output',BASE_CONFIG,'--run-name','a2d_base_only'],cwd=MOBILE_REPO)
run([sys.executable,'scripts/prepare_monodetr_a2d_gate.py','--monodetr-repo',MONODETR_REPO,'--base-config',BASE_CONFIG,'--a2-selection',A2_SELECTION,'--output-root',OUTPUT_ROOT,'--gate-epochs','5','--learning-rate','1e-5','--seed','20268'],cwd=MOBILE_REPO)
MANIFEST=OUTPUT_ROOT/'a2d_gate_manifest.json'; manifest=json.loads(MANIFEST.read_text())
assert list(manifest['variants'])==['control_w1_0','pedbox_w1_5','pedbox_w2_0','pedbox_w2_5']
assert manifest['distillation_enabled'] is False and manifest['temperature_scaling_enabled'] is False
assert manifest['random_seed']==20268 and manifest['random_seed']**2 <= 2**32-1
print(json.dumps(manifest,indent=2))


In [ ]:
# Train each five-epoch branch; completed branches are skipped on rerun.
LOG_DIR=OUTPUT_ROOT/'colab_logs'; LOG_DIR.mkdir(parents=True,exist_ok=True)
for name,variant in manifest['variants'].items():
    checkpoint=Path(variant['run_dir'])/'checkpoint_epoch_5.pth'
    if checkpoint.is_file(): print('cached',name,checkpoint); continue
    log_path=LOG_DIR/f'{name}.log'; print('training',name,'log=',log_path,flush=True)
    with log_path.open('w',encoding='utf-8',buffering=1) as log:
        process=subprocess.Popen([sys.executable,'-u','tools/train_val.py','--config',variant['config']],cwd=MONODETR_REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout: print(line,end='',flush=True); log.write(line)
        code=process.wait()
    if code: raise RuntimeError(f'{name} exited {code}; log={log_path}')
    if not checkpoint.is_file(): raise FileNotFoundError(checkpoint)


In [ ]:
# Complete AP + nearby-recall + corrected M44 comparison; this alone authorizes or rejects a full run.
run([sys.executable,'-u','scripts/evaluate_monodetr_a2d_gate.py','--mobile-repo',MOBILE_REPO,'--monodetr-repo',MONODETR_REPO,'--manifest',MANIFEST,'--dataset-root',DATASET_ROOT,'--split-dir',SPLIT_DIR],cwd=MOBILE_REPO)
report=json.loads((OUTPUT_ROOT/'a2d_gate_comparison.json').read_text()); print(json.dumps(report,indent=2))
import pandas as pd
display(pd.read_csv(OUTPUT_ROOT/'a2d_gate_comparison.csv'))
print('Full run authorized:',report['full_run_authorized'],'selected:',report['selected'])
